# Explore: features, labels, predictions

For looking only. Every number here is computed by code in `src/heatwave`; this notebook just loads the outputs under `data/` and shows them. Nothing here is used by the pipeline (reference.md §5.1).

Needs the local data (`uv run heatwave hub-pull`). Open with the project's `.venv` kernel.

Dates: **t** = as-of date (newest data), **s = t + 3** = issue date, **s + h** = valid date (lead h = 1–10).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from heatwave.config import REPO_ROOT
from heatwave.features.build import FEATURES
from heatwave.validation import metrics

DATA = REPO_ROOT / "data"
pd.set_option("display.max_columns", 60, "display.width", 200)
districts = pd.read_parquet(DATA / "reference" / "districts.parquet")[["district_id", "name"]]
name = dict(zip(districts.district_id, districts.name, strict=True))

## 1. Labels
One row per district and day. `is_heatwave` is the label used for training (IMD plains rule on bias-corrected POWER Tmax); `_imd` and `_pct` are the sensitivity labels.

In [ ]:
labels = pd.read_parquet(DATA / "labels" / "labels_daily.parquet")
labels["obs_date"] = pd.to_datetime(labels["obs_date"])
labels.sample(5, random_state=0)

In [ ]:
# Heatwave days per season (March-July), averaged over districts
season = labels[labels.obs_date.dt.month.between(3, 7)]
per_year = season.groupby(season.obs_date.dt.year)["is_heatwave"].mean() * 100
ax = per_year.plot(kind="bar", figsize=(12, 3), color="tab:red")
ax.set(xlabel="", ylabel="% of district-days", title="Heatwave rate, March-July")
ax.tick_params(axis="x", labelsize=7)

In [ ]:
episodes = pd.read_parquet(DATA / "labels" / "episodes.parquet")
episodes.sort_values("district_days", ascending=False).head(10)

## 2. Features
As-of features at date t (development years, 1981-2020). The dictionary is `docs/features.md`.

In [ ]:
pd.DataFrame([{"feature": f.name, "group": f.group} for f in FEATURES]).groupby("group")[
    "feature"
].apply(", ".join).to_frame()

In [ ]:
feats = pd.read_parquet(DATA / "features" / "development.parquet")
feats["asof_date"] = pd.to_datetime(feats["asof_date"])
feats.describe().T.round(2)

In [ ]:
# One district and season: Tmax, its anomaly, and the heatwave days
DISTRICT, YEAR = districts.district_id.iloc[0], 2010
f = feats[(feats.district_id == DISTRICT) & (feats.asof_date.dt.year == YEAR)].set_index(
    "asof_date"
)
lab = labels[(labels.district_id == DISTRICT) & (labels.obs_date.dt.year == YEAR)].set_index(
    "obs_date"
)
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(f.index, f.tmax, label="Tmax (POWER)", lw=1)
ax.plot(lab.index, lab.tmax_used, label="Tmax used by the label", lw=1)
ax.plot(lab.index, lab.normal_tmax, label="normal", color="grey", ls="--", lw=1)
hw = lab[lab.is_heatwave == 1]
ax.scatter(hw.index, hw.tmax_used, color="red", s=12, zorder=3, label="heatwave day")
ax.set(
    title=f"{name[DISTRICT]} {YEAR}",
    ylabel="degC",
    xlim=(pd.Timestamp(f"{YEAR}-03-01"), pd.Timestamp(f"{YEAR}-07-31")),
)
ax.legend(fontsize=8)

## 3. Out-of-fold predictions
Walk-forward predictions for 2006-2020: every row was predicted by a model that never saw its year. M0 = climatology, M1 = persistence, M2 = logistic regression, M3 = LightGBM (once its run has finished).

In [ ]:
oof = pd.read_parquet(DATA / "oof" / "baselines_lead.parquet")
m3_file = DATA / "oof" / "m3_default_lead.parquet"
if m3_file.exists():
    m3 = pd.read_parquet(m3_file, columns=["district_id", "issue_date", "lead", "p_M3"])
    oof = oof.merge(m3, on=["district_id", "issue_date", "lead"], how="left")
models = [c[2:] for c in oof.columns if c.startswith("p_")]
print(models)
oof.sample(5, random_state=0)

In [ ]:
# Brier skill score vs climatology by lead (point estimates; the reports have intervals)
def bss(d):
    bs0 = ((d.p_M0 - d.y) ** 2).mean()
    return pd.Series({m: 1 - ((d[f"p_{m}"] - d.y) ** 2).mean() / bs0 for m in models if m != "M0"})


skill = pd.concat(
    {"all": oof.groupby("lead").apply(bss), "onset": oof[oof.onset].groupby("lead").apply(bss)},
    axis=1,
)
skill.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, subset in zip(axes, ["all", "onset"], strict=True):
    skill[subset].plot(ax=ax, marker="o")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set(title=f"BSS vs M0 ({subset})", xlabel="lead (days)")

In [ ]:
# Reliability: mean forecast vs observed frequency (10 equal-count bins)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for m in models:
    r = metrics.reliability(oof, m, 10)
    ax.plot(r.forecast, r.observed, marker="o", label=m)
ax.plot([0, 1], [0, 1], color="grey", ls="--", lw=0.8)
top = oof[[f"p_{m}" for m in models]].quantile(0.999).max()
ax.set(
    xlim=(0, top),
    ylim=(0, top),
    xlabel="forecast",
    ylabel="observed",
    title="Reliability, all leads",
)
ax.legend()

In [ ]:
# One district and season: forecasts at lead 3 against what happened
d = oof[(oof.district_id == DISTRICT) & (oof.year == 2010) & (oof.lead == 3)].set_index(
    "valid_date"
)
fig, ax = plt.subplots(figsize=(12, 3.5))
for m in models:
    ax.plot(d.index, d[f"p_{m}"], label=m, lw=1)
ax.scatter(
    d.index[d.y == 1], [1.02] * int(d.y.sum()), color="red", marker="|", s=80, label="heatwave day"
)
ax.set(title=f"{name[DISTRICT]} 2010, lead 3", ylabel="P(heatwave day)", ylim=(0, 1.08))
ax.legend(fontsize=8, ncol=5)

In [ ]:
# Mean forecast vs observed rate per year (the hot 2010s are under-forecast)
by_year = oof.groupby("year")[["y", *[f"p_{m}" for m in models]]].mean() * 100
by_year.round(2)